## Sample Encoding

In [22]:
!python --version

Python 3.12.11


In [23]:
!which python

/Users/tom-18881/Documents/projects/my_llm/.venv/bin/python


In [24]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.15.0
tiktoken version: 0.14.0


In [25]:
import os
import requests

file_path = "the-verdict.txt"

if not os.path.exists(file_path):
    url = (
        "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [26]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("char length:", len(raw_text))
print(raw_text[:99])

char length: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [27]:
import re

text = "Hello, world. This is Tom."
re.split(r'(\s)', text)

['Hello,', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'Tom.']

In [28]:
result = re.split(r'([,..]|\s)', text)
result

['Hello',
 ',',
 '',
 ' ',
 'world',
 '.',
 '',
 ' ',
 'This',
 ' ',
 'is',
 ' ',
 'Tom',
 '.',
 '']

In [29]:
result = [item for item in result if item.strip()]
result

['Hello', ',', 'world', '.', 'This', 'is', 'Tom', '.']

In [30]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
result

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']

In [31]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
preprocessed[:30]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius',
 '--',
 'though',
 'a',
 'good',
 'fellow',
 'enough',
 '--',
 'so',
 'it',
 'was',
 'no',
 'great',
 'surprise',
 'to',
 'me',
 'to',
 'hear',
 'that',
 ',',
 'in']

In [32]:
len(preprocessed)

4690

In [33]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

vocab_size

1130

In [34]:
vocab = {token: idx for idx, token in enumerate(all_words)}

In [35]:
list(vocab.items())[:50]

[('!', 0),
 ('"', 1),
 ("'", 2),
 ('(', 3),
 (')', 4),
 (',', 5),
 ('--', 6),
 ('.', 7),
 (':', 8),
 (';', 9),
 ('?', 10),
 ('A', 11),
 ('Ah', 12),
 ('Among', 13),
 ('And', 14),
 ('Are', 15),
 ('Arrt', 16),
 ('As', 17),
 ('At', 18),
 ('Be', 19),
 ('Begin', 20),
 ('Burlington', 21),
 ('But', 22),
 ('By', 23),
 ('Carlo', 24),
 ('Chicago', 25),
 ('Claude', 26),
 ('Come', 27),
 ('Croft', 28),
 ('Destroyed', 29),
 ('Devonshire', 30),
 ('Don', 31),
 ('Dubarry', 32),
 ('Emperors', 33),
 ('Florence', 34),
 ('For', 35),
 ('Gallery', 36),
 ('Gideon', 37),
 ('Gisburn', 38),
 ('Gisburns', 39),
 ('Grafton', 40),
 ('Greek', 41),
 ('Grindle', 42),
 ('Grindles', 43),
 ('HAD', 44),
 ('Had', 45),
 ('Hang', 46),
 ('Has', 47),
 ('He', 48),
 ('Her', 49)]

In [36]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {idx : string for string, idx in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [ self.str_to_int[string] for string in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[idx] for idx in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [37]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
ids

[1,
 56,
 2,
 850,
 988,
 602,
 533,
 746,
 5,
 1126,
 596,
 5,
 1,
 67,
 7,
 38,
 851,
 1108,
 754,
 793,
 7]

In [38]:
rev_vocab = {idx : string for string, idx in vocab.items()}
for idx in ids:
    print(idx, rev_vocab[idx])

1 "
56 It
2 '
850 s
988 the
602 last
533 he
746 painted
5 ,
1126 you
596 know
5 ,
1 "
67 Mrs
7 .
38 Gisburn
851 said
1108 with
754 pardonable
793 pride
7 .


In [39]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

Adding special context tokens 

In [40]:
# hello is unknown token, it will cause error

# tokenizer = SimpleTokenizerV1(vocab)

# text = "Hello, do you like tea. Is this-- a test?"

# tokenizer.encode(text)

In [41]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [42]:
list(vocab.items())[-5:]

[('younger', 1127),
 ('your', 1128),
 ('yourself', 1129),
 ('<|endoftext|>', 1130),
 ('<|unk|>', 1131)]

In [43]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [44]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))
text

'Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.'

In [45]:
ids = tokenizer.encode(text)
ids

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [46]:
tokenizer.decode(ids)

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'